# Reproducible ERα QSAR and annotation prediction
### A fixed RDKit16 research candidate, from public source tables to six Random Forests

**Question:** can a small, explicitly defined set of molecular descriptors predict the supplied activity and five binary annotations?

This notebook replays one audited research adaptation. It is not a reproduction of the paper's DFFS/stacking pipeline, a verified antagonist model, a clinical-safety model, or a therapeutic index. The 395-row holdout **has already been evaluated and is now used evidence**. Re-running this notebook checks reproduction; it does not produce a new independent validation.

**Execution state:** all 16 code cells completed in the independently accepted local CPython 3.13.16 replay. The actual outputs and execution counts are retained. Execution used sequential bare-Python cells, **not native Jupyter/Colab front-end validation**. The small historical result table below remains a labeled record of the original audit, not a code-cell output. All code, source identifiers, hashes and scientific settings are included here; there are no private file dependencies. The acknowledgment shown in the code is the one used for the authorized replay. Re-running elsewhere is still reproduction of used evidence. This is a fixed source-data replay, **not an arbitrary-molecule prediction interface or an explorer integration**.

Read the short explanations, then follow the cells in order: source tables → fixed split → named descriptors → six fixed fits → predictions → evaluation. There is no hyperparameter search, activity gate, calibration, feature ranking or composite score.


## 1. What the source values mean

Cao and Liu report IC50 measurements associated with ERα. We retain the original `IC50_nM` and source-provided `pIC50`; row-level assay identity and a homogeneous functional-antagonist mechanism are not established.

\[
\mathrm{pIC50}=-\log_{10}(\mathrm{IC50\ in\ mol/L})=9-\log_{10}(\mathrm{IC50\ in\ nM}).
\]

For example, 100 nM corresponds to pIC50 = 7. The conversion is checked, not used to replace or round the supplied target. `IC50_nM` never enters the predictors because it encodes the same outcome.

The five labels are **source-supplied annotations**, not verified measurements. Their experimental/computational/mixed origins, assay or prediction tools, and numerical thresholds remain unresolved.

| Raw column | Meaning of 1 in the original contest codebook | Meaning of 0 |
|---|---|---|
| Caco-2 | Good intestinal epithelial permeability | Poor permeability |
| CYP3A4 | Metabolized by CYP3A4 (substrate/metabolism annotation) | Not metabolized |
| hERG | Cardiotoxic according to the annotation | Not cardiotoxic according to the annotation |
| HOB | Good human oral bioavailability | Poor bioavailability |
| MN | Genotoxic according to the annotation | Not genotoxic according to the annotation |

**CYP3A4 is not an inhibition label. hERG and MN are harm-positive.** We do not reverse them into “desirable” classes. The original contest tables were independently compared cell-for-cell with the Figshare tables; the paper's blanket description of 1 as favorable must not override these raw label meanings.

**Sources and attribution.** Cao J, Liu Y (2026), [*Multi-objective QSAR prediction of ERα antagonists via SHAP-based interpretation*](https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0338080), PLOS ONE 21(1): e0338080. Dataset: Cao J, Liu Y, [*Investigation of the bioactivity and ADMET properties of compounds targeting the breast cancer receptor ERα*, Figshare v2](https://doi.org/10.6084/m9.figshare.30459479.v2), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Label definitions: [2021 graduate mathematical-modeling contest, problem D and attachments](https://www.shumo.com/wiki/lib/exe/fetch.php?media=npmcm:npmcm2021d.7z). This adaptation changes feature generation, models, splitting and reporting; it does not imply endorsement by the authors.


## 2. Why this candidate, and what was already observed

RDKit16 was selected for an explicit, reproducible SMILES-to-feature calculation, **not for the highest development accuracy**. In the separate development experiments, its activity RMSE was 0.780979 on random folds and 0.914901 on scaffold folds, versus 0.713656 and 0.854901 for the frozen source729 RF benchmark. Its annotation balanced accuracy was below source DFFS40 for all five endpoints. Those are historical comparisons, not extra models fitted here; the source729 columns are never mixed into this notebook.

**Previously audited results, not execution output from this notebook:**

| Target / prespecified metric | Fixed RDKit16 RF | Development-only constant comparator |
|---|---:|---:|
| Source pIC50 / RMSE | 0.769016 | 1.405972 |
| Caco-2 / balanced accuracy | 0.883824 | 0.500000 |
| CYP3A4 / balanced accuracy | 0.899050 | 0.500000 |
| hERG / balanced accuracy | 0.894954 | 0.500000 |
| HOB / balanced accuracy | 0.833028 | 0.500000 |
| MN / balanced accuracy | 0.872281 | 0.500000 |

All denominators are 395. Activity MAE was 0.563541 and R² was 0.700485. The complete audited reference metrics are included as explicit constants near the evaluation cell; no score is invented or filled into a code-cell output.

There were no overlapping exact isomeric graphs, but **287/395 holdout rows share development scaffolds** (122 shared nonempty scaffolds). This mostly assesses same-source interpolation. Five holdout structures had appeared in a molecule-only fixture, and earlier source-integrity checks had read the complete labeled table. The original staged seal prevented development/scoring use of holdout outcomes, not all historical source access. This replay makes no outcome-blinding claim.


## 3. Runtime and execution boundary

The accepted scientific runtime is **CPython 3.13.16, Linux x86_64**, RDKit **2026.9.1**, NumPy **2.5.4**, SciPy **1.18.1**, scikit-learn **1.9.1**, joblib **1.6.0**, and threadpoolctl **3.7.0**. The complete original package/version/wheel-hash lock is retained below, including packages not used by these six RFs. A notebook frontend/kernel is an additional interface dependency, not part of the original modeling lock.

Use a fresh kernel in that environment. This notebook **never installs packages**, updates a runtime, deletes files, accesses credentials or queries a changing chemical API. If a required version is unavailable, stop; do not silently substitute it. The wheel hashes are Linux CP313 artifact hashes, not a claim of cross-platform binary equivalence. Matching version strings alone does not prove installed payload identity; preserve the hash-checked installation receipt.

The original audited run used stricter OS process isolation and checked runtime bytes. This portable notebook reproduces the scientific recipe, not that security sandbox. It writes only into a new relative run directory, refuses overwrite, records warnings, and stops on source/schema/nonfinite failures. It cannot make rerunning a used holdout scientifically fresh.


In [1]:
import csv, hashlib, importlib.metadata, io, json, math, os, platform
import posixpath, re, sys, warnings, zipfile
from pathlib import Path
from urllib.request import Request, urlopen
from xml.etree import ElementTree as ET

# Leave blank during review. Change only after an exact-replay authorization.
REPLAY_ACKNOWLEDGMENT = "REPLAY_FIXED_USED_HOLDOUT_NO_RETUNING"
REQUIRED_ACKNOWLEDGMENT = "REPLAY_FIXED_USED_HOLDOUT_NO_RETUNING"
RUN_DIR = Path("eralpha_rdkit16_replay_001")

def require_replay():
    if REPLAY_ACKNOWLEDGMENT != REQUIRED_ACKNOWLEDGMENT:
        raise RuntimeError("Replay is disabled. Review the notebook and obtain authorization first.")

def digest_bytes(data):
    return hashlib.sha256(data).hexdigest()

def ordered_id_hash(values):
    return digest_bytes(("\n".join(values)+"\n").encode("ascii"))

def write_json(path, obj):
    with Path(path).open("x", encoding="utf-8") as handle:
        json.dump(obj, handle, indent=2, allow_nan=False)
        handle.write("\n")

def write_csv(path, rows):
    with Path(path).open("x", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


In [2]:
PACKAGE_LOCK = {'cloudpickle': '3.1.2',
 'joblib': '1.6.0',
 'lightgbm': '4.7.0',
 'narwhals': '2.27.1',
 'numpy': '2.5.4',
 'nvidia-nccl-cu13': '2.32.3',
 'pandas': '3.1.0',
 'pillow': '13.0.0',
 'pip': '25.0.1',
 'python-dateutil': '2.9.0.post0',
 'rdkit': '2026.9.1',
 'scikit-learn': '1.9.1',
 'scipy': '1.18.1',
 'six': '1.17.0',
 'threadpoolctl': '3.7.0',
 'xgboost': '3.4.1'}
WHEEL_SHA256 = {'cloudpickle': '9acb47f6afd73f60dc1df93bb801b472f05ff42fa6c84167d25cb206be1fbf4a',
 'joblib': '3dbbf9f6e4b592a2357b854608e980fe6390d131d7a82f011a377ef2ebef7aba',
 'lightgbm': 'd23e922acd891e77212e4d0fbcee9ba973c96dee479491341d05ba595357ebb7',
 'narwhals': 'd057df13f5852b8e157596e82eb5e955fad267425df5e420e0ee9863da483b31',
 'numpy': '6ffa07666f8da0eef81d149934a626d0d95fbd6838432a33e66245423a9062c0',
 'nvidia-nccl-cu13': '1459723080ac889d73a26edfa3e04383a7928ab31ac8f0ec43b3ea9548b04ff3',
 'pandas': 'de6d0fc96e8551b83f400bf212611a0b3a2b389594d20a2540dacc61b21e7bce',
 'pillow': '5f56499c1a74d2b862beb0c33b8016f99d095858455df9d21583bbbc1c15b582',
 'pip': 'c46efd13b6aa8279f33f2864459c8ce587ea6a1a59ee20de055868d8f7688f7f',
 'python-dateutil': 'a8b2bc7bffae282281c8140a97d3aa9c14da0b136dfe83f850eea9a5f7470427',
 'rdkit': '6b4b8b814d2190744c812c1f0b0d7fe4f029ade02ca444ed584261a254624ae9',
 'scikit-learn': '55e79d6e9b0923f1a978179822bd43d7f5543f45e970a00fe861f43486380aba',
 'scipy': 'fdaf5ea890a6183d0565f51a61799d67081bd5b1cf03c5f4b3fd3732108625c9',
 'six': '4721f391ed90541fddacab5acf947aa0d3dc7d27b2e1e8eda2be8970586c3274',
 'threadpoolctl': 'cd8b60b5641b45c67bbf73c64c843235fc2d8a480c87389f52f5dbee893b86be',
 'xgboost': '6adf2afa396da2ae8ed30295b50b99d4712eed9a6e0ce6cfe069290e4335e51f'}

# This generates text only: installing these packages is a separate operation.
REQUIREMENTS_LOCK = "\n".join(
    f"{name}=={version} --hash=sha256:{WHEEL_SHA256[name]}"
    for name, version in PACKAGE_LOCK.items()) + "\n"
print(REQUIREMENTS_LOCK)


cloudpickle==3.1.2 --hash=sha256:9acb47f6afd73f60dc1df93bb801b472f05ff42fa6c84167d25cb206be1fbf4a
joblib==1.6.0 --hash=sha256:3dbbf9f6e4b592a2357b854608e980fe6390d131d7a82f011a377ef2ebef7aba
lightgbm==4.7.0 --hash=sha256:d23e922acd891e77212e4d0fbcee9ba973c96dee479491341d05ba595357ebb7
narwhals==2.27.1 --hash=sha256:d057df13f5852b8e157596e82eb5e955fad267425df5e420e0ee9863da483b31
numpy==2.5.4 --hash=sha256:6ffa07666f8da0eef81d149934a626d0d95fbd6838432a33e66245423a9062c0
nvidia-nccl-cu13==2.32.3 --hash=sha256:1459723080ac889d73a26edfa3e04383a7928ab31ac8f0ec43b3ea9548b04ff3
pandas==3.1.0 --hash=sha256:de6d0fc96e8551b83f400bf212611a0b3a2b389594d20a2540dacc61b21e7bce
pillow==13.0.0 --hash=sha256:5f56499c1a74d2b862beb0c33b8016f99d095858455df9d21583bbbc1c15b582
pip==25.0.1 --hash=sha256:c46efd13b6aa8279f33f2864459c8ce587ea6a1a59ee20de055868d8f7688f7f
python-dateutil==2.9.0.post0 --hash=sha256:a8b2bc7bffae282281c8140a97d3aa9c14da0b136dfe83f850eea9a5f7470427
rdkit==2026.9.1 --hash=sha256:6b4b8b

In [3]:
require_replay()
if platform.python_version() != "3.13.16" or platform.system() != "Linux" or platform.machine() != "x86_64":
    raise RuntimeError("Use the accepted CPython 3.13.16 Linux x86_64 runtime.")
for name, version in PACKAGE_LOCK.items():
    if importlib.metadata.version(name) != version:
        raise RuntimeError(f"Dependency mismatch: {name}; no automatic replacement permitted.")
if any(name in sys.modules for name in ["numpy", "sklearn", "rdkit"]):
    raise RuntimeError("Start a fresh kernel so thread settings apply before scientific imports.")
for name in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ[name] = "1"

RUN_DIR.mkdir(exist_ok=False)  # Never overwrite an earlier attempt, even a failed one.
(RUN_DIR / "requirements.lock.txt").write_text(REQUIREMENTS_LOCK)
WARNING_LOG = []
caught, import_error = [], None
try:
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        import numpy as np
        import joblib
        from rdkit import Chem, rdBase
        from rdkit.Chem import Descriptors, rdMolDescriptors, rdPartialCharges
        from rdkit.Chem.Scaffolds import MurckoScaffold
        from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
        from sklearn.model_selection import train_test_split
        from threadpoolctl import threadpool_limits, threadpool_info
except BaseException as exc:
    import_error = {"type":type(exc).__name__,"message":str(exc)}
    raise
finally:
    WARNING_LOG.extend({"stage":"imports", "category":w.category.__name__, "message":str(w.message)} for w in caught)
    write_json(RUN_DIR / "import_status.json",{"status":"failed" if import_error else "complete",
        "exception":import_error,"warnings":WARNING_LOG,"model_fits":0})
write_json(RUN_DIR / "runtime.json", {"python":platform.python_version(),
    "platform":platform.platform(), "packages":PACKAGE_LOCK, "rdkit_build":rdBase.rdkitBuild,
    "n_jobs":1, "warnings":WARNING_LOG, "holdout_status":"already_used"})
print("Runtime versions matched; this is not installed-payload verification.")


Runtime versions matched; this is not installed-payload verification.


## 4. Retrieve only the fixed source files

We fetch two immutable Figshare file IDs from dataset version 2 and check their sizes and SHA-256 hashes **before parsing**. A changed download fails rather than becoming a new dataset. For an offline replay, place copies of these same two public workbooks in the explicitly named relative folder `eralpha_public_source_cache/`; cached bytes must pass the identical checks and the acquisition route is recorded. This optional cache is not a private input dependency. The 729 supplied descriptors and their workbook are unnecessary for this candidate.

The small reader below handles the plain OOXML tables in these exact files with the Python standard library. It rejects formulas, external workbook links, unsupported cell types, duplicate coordinates and non-contiguous rows. No spreadsheet macros are executed. Blank outcomes remain missing, never zero. The two `training` sheets contain 1,974 labeled records; their `test` sheets contain 50 unlabeled records and are not a valid accuracy test set.


In [4]:
SOURCES = [{'file': 'Compound_Activity.xlsx',
  'file_id': 59100485,
  'source_url': 'https://ndownloader.figshare.com/files/59100485',
  'size': 93543,
  'sha256': 'a4ef5fe918d552aa497e1f44ae55c60a00fd1c16f712d265ab6495c7c96b6647'},
 {'file': 'Compound_ADMET.xlsx',
  'file_id': 59100224,
  'source_url': 'https://ndownloader.figshare.com/files/59100224',
  'size': 86866,
  'sha256': '8e5a569ebcaa6289ecc9de051eb0d10cf89d0748396725c948ef467a14003e61'}]

SOURCE_CACHE = Path("eralpha_public_source_cache")  # Optional public-file cache, not required online.
SOURCE_ACQUISITION = []

def fetch_verified(spec, directory, cache_dir):
    cached = cache_dir / spec["file"]
    if cache_dir.is_symlink() or cached.is_symlink():
        raise ValueError("Do not follow source-cache symlinks")
    if cached.is_file():
        if cached.stat().st_size != spec["size"]:
            raise ValueError(f"Cached source size changed: {spec['file']}")
        data, route = cached.read_bytes(), "hash_checked_local_public_copy"
    else:
        request = Request(spec["source_url"], headers={"User-Agent":"Reproducible-ERalpha-research/1.0"})
        with urlopen(request, timeout=60) as response:
            if not response.geturl().startswith("https://"):
                raise ValueError("Refuse a non-HTTPS source redirect")
            data = response.read(spec["size"] + 1)
        route = "immutable_public_HTTPS_file_ID"
    if len(data) != spec["size"] or digest_bytes(data) != spec["sha256"]:
        raise ValueError(f"Source bytes changed: {spec['file']}")
    path = directory / spec["file"]
    with path.open("xb") as handle:
        handle.write(data)
    SOURCE_ACQUISITION.append({"file":spec["file"],"route":route,"sha256":spec["sha256"]})
    return path

require_replay()
source_paths = {s["file"]:fetch_verified(s, RUN_DIR, SOURCE_CACHE) for s in SOURCES}
write_json(RUN_DIR / "source_manifest.json", {"dataset_doi":"10.6084/m9.figshare.30459479.v2",
    "license":"CC BY 4.0", "authors":["Jinhui Cao","Yanli Liu"], "files":SOURCES,
    "acquisition":SOURCE_ACQUISITION})


In [5]:
def read_plain_xlsx(path, expected_columns):
    ns = {"s":"http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
    rel_ns = "http://schemas.openxmlformats.org/package/2006/relationships"
    rid_ns = "http://schemas.openxmlformats.org/officeDocument/2006/relationships"
    result = {}
    with zipfile.ZipFile(path) as archive:
        if any("externalLink" in name or name.endswith("vbaProject.bin") for name in archive.namelist()):
            raise ValueError("Unexpected workbook links or macros")
        rels = ET.fromstring(archive.read("xl/_rels/workbook.xml.rels"))
        if any(r.get("TargetMode") == "External" for r in rels):
            raise ValueError("External workbook relationship")
        targets = {r.get("Id"):r.get("Target") for r in rels}
        shared = []
        if "xl/sharedStrings.xml" in archive.namelist():
            strings = ET.fromstring(archive.read("xl/sharedStrings.xml"))
            shared = ["".join(si.itertext()) for si in strings]
        workbook = ET.fromstring(archive.read("xl/workbook.xml"))
        for sheet in workbook.findall("s:sheets/s:sheet", ns):
            target = targets[sheet.get("{"+rid_ns+"}id")]
            member = target.lstrip("/") if target.startswith("/") else posixpath.normpath("xl/"+target)
            if not member.startswith("xl/") or ".." in member.split("/"):
                raise ValueError("Invalid worksheet member path")
            tree = ET.fromstring(archive.read(member))
            rows = []
            for expected_row, row in enumerate(tree.findall("s:sheetData/s:row", ns), 1):
                if int(row.get("r")) != expected_row:
                    raise ValueError("Missing or reordered source row")
                values = [None] * expected_columns
                seen = set()
                for cell in row.findall("s:c", ns):
                    if cell.find("s:f", ns) is not None:
                        raise ValueError("Formula cells are not accepted")
                    match = re.fullmatch(r"([A-Z]+)([1-9][0-9]*)", cell.get("r", ""))
                    if not match or int(match[2]) != expected_row:
                        raise ValueError("Invalid cell reference")
                    col = 0
                    for char in match[1]:
                        col = col*26 + ord(char)-64
                    if col < 1 or col > expected_columns or col in seen:
                        raise ValueError("Unexpected or duplicate column")
                    seen.add(col)
                    kind = cell.get("t", "n")
                    text = cell.findtext("s:v", default=None, namespaces=ns)
                    if kind == "s": value = shared[int(text)]
                    elif kind == "inlineStr": value = "".join(cell.find("s:is", ns).itertext())
                    elif kind == "n": value = None if text is None else float(text)
                    else: raise ValueError(f"Unsupported source cell type: {kind}")
                    values[col-1] = value
                rows.append(values)
            if sheet.get("name") in result:
                raise ValueError("Duplicate worksheet name")
            result[sheet.get("name")] = rows
    if list(result) != ["training", "test"]:
        raise ValueError("Unexpected sheets")
    return result


In [6]:
ENDPOINTS = ["Caco-2", "CYP3A4", "hERG", "HOB", "MN"]
OUTPUTS = ["p_raw1_Caco2", "p_raw1_CYP3A4_metabolized", "p_raw1_hERG_cardiotoxic",
           "p_raw1_HOB_good", "p_raw1_MN_genotoxic"]

def assemble_source(activity, annotation):
    records = []
    for sheet, count in [("training",1974),("test",50)]:
        a, b = activity[sheet], annotation[sheet]
        assert a[0] == ["SMILES","IC50_nM","pIC50"]
        assert b[0] == ["SMILES"] + ENDPOINTS
        assert len(a) == len(b) == count+1
        a_keys, b_keys = [r[0] for r in a[1:]], [r[0] for r in b[1:]]
        assert len(set(a_keys)) == len(set(b_keys)) == count
        assert a_keys == b_keys  # Check alignment; join explicitly by exact source SMILES.
        annotations = {r[0]:r[1:] for r in b[1:]}
        for excel_row, (smiles, ic50, pic50) in enumerate(a[1:], 2):
            labels = annotations[smiles]
            assert isinstance(smiles,str) and smiles
            if sheet == "training":
                assert all(isinstance(v,(int,float)) and math.isfinite(v) for v in [ic50,pic50]+labels)
                assert ic50 > 0 and abs(pic50-(9-math.log10(ic50))) < 1e-12
                assert all(v in (0,1) for v in labels)
                labels = [int(v) for v in labels]
            else:
                assert all(v is None for v in [ic50,pic50]+labels)
            records.append({"source_record_id":f"figshare30459479v2:{sheet}:row{excel_row}",
                "source_sheet":sheet, "source_excel_row":excel_row, "SMILES":smiles,
                "IC50_nM":ic50, "pIC50":pic50, **dict(zip(ENDPOINTS,labels))})
    assert len({r["source_record_id"] for r in records}) == 2024
    return records

require_replay()
records = assemble_source(read_plain_xlsx(source_paths["Compound_Activity.xlsx"],3),
                          read_plain_xlsx(source_paths["Compound_ADMET.xlsx"],6))
labeled = [r for r in records if r["source_sheet"] == "training"]
unlabeled = [r for r in records if r["source_sheet"] == "test"]
assert len(labeled) == 1974 and len(unlabeled) == 50
write_csv(RUN_DIR / "source_outcomes.csv", records)
print("1,974 source-labeled records; 50 unlabeled records excluded from fitting and prediction.")


1,974 source-labeled records; 50 unlabeled records excluded from fitting and prediction.


## 5. Reconstruct the split without making a new choice

The original split was an unstratified 80:20 row split with seed **338080**. We reconstruct it from source order and verify the ordered source-ID hashes. Development rows keep the exact randomized training order; holdout rows are sorted by original Excel row. Sorting the development rows would change bootstrap samples even with the same RF seed.

`source_record_id` records the dataset version, original sheet and Excel row. IDs, numeric row numbers, SMILES, all seven outcomes and support diagnostics are provenance/targets, **never numeric predictors**. No fresh split or cross-validation search is run here.


In [7]:
require_replay()
development_index, holdout_index = train_test_split(np.arange(1974), test_size=0.2,
                                                  random_state=338080, shuffle=True)
holdout_index = np.sort(holdout_index)
development = [labeled[int(i)] for i in development_index]
holdout = [labeled[int(i)] for i in holdout_index]
development_ids = [r["source_record_id"] for r in development]
holdout_ids = [r["source_record_id"] for r in holdout]
assert len(development_ids)==1579 and len(holdout_ids)==395 and not set(development_ids)&set(holdout_ids)
assert ordered_id_hash(development_ids)=="3d1407e76329e33efece5d1db8401bacf0cf1a8ba43cc4a507bf32eebef24650"
assert ordered_id_hash(holdout_ids)=="9cfd8e3cc7e6d0939459b9754e434b6889c922cd89e1639bb7211870b21402fc"
write_json(RUN_DIR / "partition_ids.json", {"development":development_ids,"holdout":holdout_ids})


## 6. Sixteen predictors, with explicit definitions

Names are version-prefixed to prevent confusion with the 729 source descriptors. The table defines the quantities; the following function shows every RDKit API call. None was selected by holdout performance.

| Exact predictor name | Meaning | Unit |
|---|---|---|
| `rdkit_2026_09_1__MolWt` | Average molecular weight, with hydrogens | Da |
| `rdkit_2026_09_1__CrippenLogP` | Wildman-Crippen lipophilicity estimate | dimensionless |
| `rdkit_2026_09_1__CrippenMR` | Wildman-Crippen molar refractivity estimate | implementation-defined reporting scale; units to be source-verified |
| `rdkit_2026_09_1__TPSA_noSandP` | Topological polar surface area using the explicit N/O-only option | Å² |
| `rdkit_2026_09_1__HBD` | Hydrogen-bond donor count by RDKit rules | count |
| `rdkit_2026_09_1__HBA` | Hydrogen-bond acceptor count by RDKit rules | count |
| `rdkit_2026_09_1__RotatableBonds_Strict` | Rotatable-bond count using the Strict definition | count |
| `rdkit_2026_09_1__RingCount` | Ring count by the pinned RDKit ring-perception implementation | count |
| `rdkit_2026_09_1__AromaticRingCount` | Aromatic-ring count under the pinned default aromaticity model | count |
| `rdkit_2026_09_1__FractionCSP3` | Fraction of carbon atoms assigned sp3 hybridization | fraction |
| `rdkit_2026_09_1__HeavyAtomCount` | Non-hydrogen atom count | count |
| `rdkit_2026_09_1__HeteroatomCount` | Heteroatom count by the pinned RDKit definition | count |
| `rdkit_2026_09_1__NetFormalCharge` | Sum of formal charges over all retained fragments | elementary-charge units |
| `rdkit_2026_09_1__GasteigerHeavyAtomMin` | Minimum heavy-atom Gasteiger partial charge; excludes implicit-H contribution | elementary-charge units |
| `rdkit_2026_09_1__GasteigerHeavyAtomMax` | Maximum heavy-atom Gasteiger partial charge; excludes implicit-H contribution | elementary-charge units |
| `rdkit_2026_09_1__GasteigerHeavyAtomSumAbs` | Sum of absolute heavy-atom Gasteiger partial charges; size-dependent, not net charge | elementary-charge units |

**Molecule policy:** parse the supplied SMILES, sanitize, remove ordinary explicit hydrogen atoms under RDKit's pinned parser rules, and retain fragments, charge, isotope and stereo information. No salt stripping, neutralization, tautomer normalization, conformer generation or pH assignment. Descriptors describe this retained graph, which may differ from the dominant species in an assay.

TPSA explicitly excludes sulfur/phosphorus contributions. Rotatable bonds use `Strict`. Gasteiger charges use **12 iterations** with parameter failures raised; summaries include only heavy-atom `_GasteigerCharge`, not the separately stored implicit-H charge. Sum of absolute partial charges is size-dependent and is not formal net charge. Crippen MR's reporting unit is deliberately left unresolved rather than guessed.

See the primary [RDKit molecular-descriptor API](https://www.rdkit.org/docs/source/rdkit.Chem.rdMolDescriptors.html) and [partial-charge API](https://www.rdkit.org/docs/source/rdkit.Chem.rdPartialCharges.html). Exact arguments below, not changing documentation defaults, define this recipe.


In [8]:
FEATURE_NAMES = ['rdkit_2026_09_1__MolWt',
 'rdkit_2026_09_1__CrippenLogP',
 'rdkit_2026_09_1__CrippenMR',
 'rdkit_2026_09_1__TPSA_noSandP',
 'rdkit_2026_09_1__HBD',
 'rdkit_2026_09_1__HBA',
 'rdkit_2026_09_1__RotatableBonds_Strict',
 'rdkit_2026_09_1__RingCount',
 'rdkit_2026_09_1__AromaticRingCount',
 'rdkit_2026_09_1__FractionCSP3',
 'rdkit_2026_09_1__HeavyAtomCount',
 'rdkit_2026_09_1__HeteroatomCount',
 'rdkit_2026_09_1__NetFormalCharge',
 'rdkit_2026_09_1__GasteigerHeavyAtomMin',
 'rdkit_2026_09_1__GasteigerHeavyAtomMax',
 'rdkit_2026_09_1__GasteigerHeavyAtomSumAbs']

PARSER = {"sanitize":True, "removeHs":True, "parseName":False,
          "allowCXSMILES":False, "strictCXSMILES":True, "debugParse":0}
WRITER = {"allBondsExplicit":False, "allHsExplicit":False, "canonical":True,
          "cleanStereo":True, "doIsomericSmiles":True, "doKekule":False,
          "doRandom":False, "ignoreAtomMapNumbers":False, "includeDativeBonds":True,
          "rootedAtAtom":-1}

def rdkit16(smiles):
    assert Chem.GetUseLegacyStereoPerception() is True
    assert Chem.GetAllowNontetrahedralChirality() is True
    parser = Chem.SmilesParserParams()
    surface = {k for k in dir(parser) if not k.startswith("_") and isinstance(getattr(parser,k),(bool,int,float,str))}
    assert surface == set(PARSER)
    for name, value in PARSER.items(): setattr(parser,name,value)
    mol = Chem.MolFromSmiles(smiles, parser)
    if mol is None: raise ValueError("Source SMILES did not parse/sanitize")
    writer = Chem.SmilesWriteParams()
    for name, value in WRITER.items(): setattr(writer,name,value)
    canonical = Chem.MolToSmiles(Chem.Mol(mol),writer)
    formula = rdMolDescriptors.CalcMolFormula(mol)
    formal_charge = sum(a.GetFormalCharge() for a in mol.GetAtoms())
    charged = Chem.Mol(mol)
    rdPartialCharges.ComputeGasteigerCharges(charged,nIter=12,throwOnParamFailure=True)
    heavy_q = []
    for atom in charged.GetAtoms():
        q, hq = float(atom.GetProp("_GasteigerCharge")), float(atom.GetProp("_GasteigerHCharge"))
        if not (math.isfinite(q) and math.isfinite(hq)): raise ValueError("Nonfinite atomic or implicit-H charge")
        if atom.GetAtomicNum()>1: heavy_q.append(q)
    if not heavy_q: raise ValueError("No heavy atoms for charge summaries")
    logp, mr = rdMolDescriptors.CalcCrippenDescriptors(mol,includeHs=True,force=True)
    values = [Descriptors.MolWt(mol),logp,mr,
        rdMolDescriptors.CalcTPSA(mol,force=True,includeSandP=False),
        rdMolDescriptors.CalcNumHBD(mol),rdMolDescriptors.CalcNumHBA(mol),
        rdMolDescriptors.CalcNumRotatableBonds(mol,rdMolDescriptors.NumRotatableBondsOptions.Strict),
        rdMolDescriptors.CalcNumRings(mol),rdMolDescriptors.CalcNumAromaticRings(mol),
        rdMolDescriptors.CalcFractionCSP3(mol),rdMolDescriptors.CalcNumHeavyAtoms(mol),
        rdMolDescriptors.CalcNumHeteroatoms(mol),formal_charge,
        min(heavy_q),max(heavy_q),sum(abs(q) for q in heavy_q)]
    if len(values)!=16 or not all(math.isfinite(float(v)) for v in values):
        raise ValueError("Incomplete/nonfinite feature row")
    scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol,includeChirality=False)
    return values, {"canonical_isomeric_smiles":canonical,"murcko_scaffold":scaffold,"formula":formula}


In [9]:
EXPECTED_FEATURE_ARRAY_SHA256 = {'development': '47a4fdd0d599ccb4811e833c8b09082fc11ed39157399489db1c56cd0950b356',
 'holdout': '71384997e8916751035bc2ebdd28029a530f1fb2869eb1b8f36ad0d5550ade2f'}

require_replay()
feature_rows, identity, status_rows = {}, {}, []
try:
    for record in labeled:  # Source order; no outcomes are passed to the calculator.
        rid = record["source_record_id"]
        values, graph = rdkit16(record["SMILES"])
        feature_rows[rid], identity[rid] = values, graph
        status_rows.append({"source_record_id":rid,"status":"complete"})
except Exception as exc:
    status_rows.append({"source_record_id":rid,"status":type(exc).__name__+": "+str(exc)})
    done = {r["source_record_id"] for r in status_rows}
    status_rows.extend({"source_record_id":r["source_record_id"],"status":"not_attempted"}
                       for r in labeled if r["source_record_id"] not in done)
    write_csv(RUN_DIR / "feature_statuses.csv",status_rows)
    raise  # No alternate parser, omitted row, zero fill or automatic retry.
write_csv(RUN_DIR / "feature_statuses.csv",status_rows)
X = np.array([feature_rows[rid] for rid in development_ids],dtype=np.float64)
H = np.array([feature_rows[rid] for rid in holdout_ids],dtype=np.float64)
assert X.shape==(1579,16) and H.shape==(395,16)
for role, matrix in [("development",X),("holdout",H)]:
    # Ordered, little-endian float64 array hashes derived from the audited matrices.
    assert digest_bytes(matrix.astype("<f8").tobytes(order="C"))==EXPECTED_FEATURE_ARRAY_SHA256[role]
    names = development_ids if role=="development" else holdout_ids
    write_csv(RUN_DIR / f"{role}_features.csv",[
        {"source_record_id":rid,**dict(zip(FEATURE_NAMES,row.tolist()))}
        for rid,row in zip(names,matrix)])
print("All 31,584 calculated feature values match the audited array hashes.")


All 31,584 calculated feature values match the audited array hashes.


## 7. Learn only from development rows

A constant descriptor cannot split a tree. We compute one exact nonconstant mask using development X alone, then apply it unchanged to both partitions. All 16 were retained in the accepted run. There is no scaling, imputation or learned feature selection.

The activity comparator is the development mean pIC50. Each annotation comparator is its development fraction of raw1 labels. These arithmetic baselines require **zero additional fitted models**. Support checks report exact graph/scaffold overlap and marginal descriptor ranges; they neither clip values nor exclude rows. Being inside every marginal range does not guarantee a supported combination or a reliable prediction.


In [10]:
DEVELOPMENT_COUNTS = {'Caco-2': [960, 619],
 'CYP3A4': [407, 1172],
 'hERG': [714, 865],
 'HOB': [1152, 427],
 'MN': [365, 1214]}

require_replay()
y_development = np.array([r["pIC50"] for r in development],dtype=float)
labels_development = np.array([[r[e] for e in ENDPOINTS] for r in development],dtype=np.int8)
assert np.isfinite(X).all() and np.isfinite(H).all() and np.isfinite(y_development).all()
for j, endpoint in enumerate(ENDPOINTS):
    assert np.bincount(labels_development[:,j],minlength=2).tolist()==DEVELOPMENT_COUNTS[endpoint]
mask = np.any(X != X[0],axis=0)
assert mask.tolist()==[True]*16
Xt, Xh = X[:,mask], H[:,mask]
activity_mean = float(np.mean(y_development))
priors = {e:float(np.mean(labels_development[:,j])) for j,e in enumerate(ENDPOINTS)}
minima, maxima = X.min(axis=0), X.max(axis=0)
development_graphs = {identity[r]["canonical_isomeric_smiles"] for r in development_ids}
development_scaffolds = {identity[r]["murcko_scaffold"] for r in development_ids} - {""}
support = []
for i,rid in enumerate(holdout_ids):
    graph = identity[rid]
    outside = np.flatnonzero((H[i]<minima)|(H[i]>maxima))
    support.append({"source_record_id":rid,
        "exact_graph_seen":graph["canonical_isomeric_smiles"] in development_graphs,
        "scaffold_seen":bool(graph["murcko_scaffold"] and graph["murcko_scaffold"] in development_scaffolds),
        "out_of_range_names":[FEATURE_NAMES[k] for k in outside]})
assert len({identity[r]["canonical_isomeric_smiles"] for r in development_ids+holdout_ids})==1974
assert sum(r["exact_graph_seen"] for r in support)==0
assert sum(r["scaffold_seen"] for r in support)==287
assert len(development_scaffolds & {identity[r]["murcko_scaffold"] for r in holdout_ids})==122
assert [(r["source_record_id"],r["out_of_range_names"]) for r in support if r["out_of_range_names"]] == [
    ("figshare30459479v2:training:row1498",["rdkit_2026_09_1__GasteigerHeavyAtomMax"])]
write_json(RUN_DIR / "support_diagnostics.json",support)
write_json(RUN_DIR / "development_preprocessing.json",{"candidate_names":FEATURE_NAMES,
    "mask":mask.tolist(),"retained_names":[n for n,k in zip(FEATURE_NAMES,mask) if k],
    "minima":minima.tolist(),"maxima":maxima.tolist(),"activity_mean":activity_mean,"raw1_priors":priors})


## 8. Exactly six fixed models

The regressor averages 300 bootstrapped decision trees fitted to supplied pIC50. The five classifiers each average 300 trees fitted to one raw binary annotation. A minimum leaf size of 2 limits single-row leaves. The regressor considers all retained features per split; classifiers use `sqrt`. No class weighting or resampling is used.

The dictionaries below are the **complete accepted parameters**, including seeds. All six models use `n_jobs=1` and native thread limits of 1. This serial choice was recorded prospectively for the original guarded final run; earlier development specifications used `n_jobs=2`. It was not an accuracy-driven change, and no parallel-runtime bitwise equivalence is claimed.

An annotation output is an **uncalibrated RF raw1 score**, not a verified probability of biological harm or clinical safety. Calls use `score >= 0.5`, including ties; no threshold is chosen from holdout outcomes.


In [11]:
RF_PARAMETERS = {'pIC50': {'n_estimators': 300,
           'criterion': 'squared_error',
           'max_depth': None,
           'min_samples_split': 2,
           'min_samples_leaf': 2,
           'min_weight_fraction_leaf': 0.0,
           'max_features': 1.0,
           'max_leaf_nodes': None,
           'min_impurity_decrease': 0.0,
           'bootstrap': True,
           'oob_score': False,
           'n_jobs': 1,
           'random_state': 338082,
           'verbose': 0,
           'warm_start': False,
           'ccp_alpha': 0.0,
           'max_samples': None,
           'monotonic_cst': None},
 'CYP3A4': {'bootstrap': True,
            'ccp_alpha': 0.0,
            'class_weight': None,
            'criterion': 'gini',
            'max_depth': None,
            'max_features': 'sqrt',
            'max_leaf_nodes': None,
            'max_samples': None,
            'min_impurity_decrease': 0.0,
            'min_samples_leaf': 2,
            'min_samples_split': 2,
            'min_weight_fraction_leaf': 0.0,
            'monotonic_cst': None,
            'n_estimators': 300,
            'n_jobs': 1,
            'oob_score': False,
            'random_state': 338102,
            'verbose': 0,
            'warm_start': False},
 'Caco-2': {'bootstrap': True,
            'ccp_alpha': 0.0,
            'class_weight': None,
            'criterion': 'gini',
            'max_depth': None,
            'max_features': 'sqrt',
            'max_leaf_nodes': None,
            'max_samples': None,
            'min_impurity_decrease': 0.0,
            'min_samples_leaf': 2,
            'min_samples_split': 2,
            'min_weight_fraction_leaf': 0.0,
            'monotonic_cst': None,
            'n_estimators': 300,
            'n_jobs': 1,
            'oob_score': False,
            'random_state': 338101,
            'verbose': 0,
            'warm_start': False},
 'HOB': {'bootstrap': True,
         'ccp_alpha': 0.0,
         'class_weight': None,
         'criterion': 'gini',
         'max_depth': None,
         'max_features': 'sqrt',
         'max_leaf_nodes': None,
         'max_samples': None,
         'min_impurity_decrease': 0.0,
         'min_samples_leaf': 2,
         'min_samples_split': 2,
         'min_weight_fraction_leaf': 0.0,
         'monotonic_cst': None,
         'n_estimators': 300,
         'n_jobs': 1,
         'oob_score': False,
         'random_state': 338104,
         'verbose': 0,
         'warm_start': False},
 'MN': {'bootstrap': True,
        'ccp_alpha': 0.0,
        'class_weight': None,
        'criterion': 'gini',
        'max_depth': None,
        'max_features': 'sqrt',
        'max_leaf_nodes': None,
        'max_samples': None,
        'min_impurity_decrease': 0.0,
        'min_samples_leaf': 2,
        'min_samples_split': 2,
        'min_weight_fraction_leaf': 0.0,
        'monotonic_cst': None,
        'n_estimators': 300,
        'n_jobs': 1,
        'oob_score': False,
        'random_state': 338105,
        'verbose': 0,
        'warm_start': False},
 'hERG': {'bootstrap': True,
          'ccp_alpha': 0.0,
          'class_weight': None,
          'criterion': 'gini',
          'max_depth': None,
          'max_features': 'sqrt',
          'max_leaf_nodes': None,
          'max_samples': None,
          'min_impurity_decrease': 0.0,
          'min_samples_leaf': 2,
          'min_samples_split': 2,
          'min_weight_fraction_leaf': 0.0,
          'monotonic_cst': None,
          'n_estimators': 300,
          'n_jobs': 1,
          'oob_score': False,
          'random_state': 338103,
          'verbose': 0,
          'warm_start': False}}
FROZEN_PROTOCOL_SHA256 = 'cb26d3242d9c2dcd7b7d87f054e9285edad64934a7a974f07804baaf55800fe7'

for name, settings in RF_PARAMETERS.items():
    estimator = RandomForestRegressor(**settings) if name=="pIC50" else RandomForestClassifier(**settings)
    assert estimator.get_params()==settings and settings["n_jobs"]==1
print(json.dumps(RF_PARAMETERS,indent=2))


{
  "pIC50": {
    "n_estimators": 300,
    "criterion": "squared_error",
    "max_depth": null,
    "min_samples_split": 2,
    "min_samples_leaf": 2,
    "min_weight_fraction_leaf": 0.0,
    "max_features": 1.0,
    "max_leaf_nodes": null,
    "min_impurity_decrease": 0.0,
    "bootstrap": true,
    "oob_score": false,
    "n_jobs": 1,
    "random_state": 338082,
    "verbose": 0,
    "warm_start": false,
    "ccp_alpha": 0.0,
    "max_samples": null,
    "monotonic_cst": null
  },
  "CYP3A4": {
    "bootstrap": true,
    "ccp_alpha": 0.0,
    "class_weight": null,
    "criterion": "gini",
    "max_depth": null,
    "max_features": "sqrt",
    "max_leaf_nodes": null,
    "max_samples": null,
    "min_impurity_decrease": 0.0,
    "min_samples_leaf": 2,
    "min_samples_split": 2,
    "min_weight_fraction_leaf": 0.0,
    "monotonic_cst": null,
    "n_estimators": 300,
    "n_jobs": 1,
    "oob_score": false,
    "random_state": 338102,
    "verbose": 0,
    "warm_start": false
  },
  "

In [12]:
require_replay()
# The marker is consumed before any fit. A failed or interrupted attempt is preserved.
write_json(RUN_DIR / "fit_started.json",{"protocol_sha256":FROZEN_PROTOCOL_SHA256,
    "model_count":6,"holdout_status":"already_used_replay","tuning":False})
predictions = [{"source_record_id":rid,"holdout_order":i} for i,rid in enumerate(holdout_ids)]
completed_fits = 0
attempted_fits = 0
model_hashes = {}
stage, current_model, fit_error = "initial_checks", None, None
try:
    with threadpool_limits(limits=1):
        assert all(v.get("num_threads",1)==1 for v in threadpool_info())
        for j,key in enumerate(["pIC50"]+ENDPOINTS):
            current_model, stage = key, "construct_model"
            model = RandomForestRegressor(**RF_PARAMETERS[key]) if j==0 else RandomForestClassifier(**RF_PARAMETERS[key])
            target = y_development if j==0 else labels_development[:,j-1]
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")
                try:
                    stage = "fit"
                    attempted_fits += 1
                    model.fit(Xt,target)
                    completed_fits += 1
                    stage = "persist_model"
                    name = f"model_{j}_{key}.joblib"
                    joblib.dump({"model":model,"mask":mask,"source_feature_names":FEATURE_NAMES,
                        "retained_feature_names":[n for n,k in zip(FEATURE_NAMES,mask) if k],
                        "target_name":key,"protocol_sha256":FROZEN_PROTOCOL_SHA256},RUN_DIR/name,compress=3)
                    model_hashes[name] = digest_bytes((RUN_DIR/name).read_bytes())
                    stage = "predict"
                    if j==0:
                        values = np.asarray(model.predict(Xh),dtype=float)
                        assert values.shape==(395,) and np.isfinite(values).all()
                        for row,value in zip(predictions,values):
                            row["predicted_source_pIC50"] = float(value)
                            row["comparator_mean_source_pIC50"] = activity_mean
                    else:
                        assert set(model.classes_.tolist())=={0,1}
                        probabilities = np.asarray(model.predict_proba(Xh),dtype=float)
                        assert probabilities.shape==(395,2) and np.isfinite(probabilities).all()
                        assert np.all((probabilities>=0)&(probabilities<=1))
                        assert np.allclose(probabilities.sum(axis=1),1,atol=1e-12,rtol=0)
                        values = probabilities[:,int(np.flatnonzero(model.classes_==1)[0])]
                        output = OUTPUTS[j-1]
                        for row,value in zip(predictions,values):
                            row[output] = float(value)
                            row["raw_call_"+key] = int(value>=0.5)
                            row["comparator_prior_"+output] = priors[key]
                            row["comparator_raw_call_"+key] = int(priors[key]>=0.5)
                finally:
                    WARNING_LOG.extend({"stage":key,"category":w.category.__name__,"message":str(w.message)} for w in caught)
            print(f"Persisted {key}: {completed_fits}/6 fits")
    assert completed_fits==attempted_fits==6
    stage = "persist_predictions"
    write_csv(RUN_DIR / "holdout_predictions.csv",predictions)
    write_json(RUN_DIR / "model_hashes.json",model_hashes)
    stage = "fit_predict_complete"
except BaseException as exc:
    fit_error = {"type":type(exc).__name__,"message":str(exc)}
    raise
finally:
    write_json(RUN_DIR / "fit_attempt_status.json",{"attempted_fits":attempted_fits,
        "completed_fits":completed_fits,"stage":stage,"current_model":current_model,
        "exception":fit_error,"warnings":WARNING_LOG,"model_hashes":model_hashes,
        "partial_predictions":predictions if fit_error else None,
        "holdout_status":"already_used_replay"})

# Exact recorded serialization checks prediction reproduction before metric calculation.
prediction_sha256 = digest_bytes((RUN_DIR / "holdout_predictions.csv").read_bytes())
assert prediction_sha256=="baef49de75f18b21f6320900218481beb3e0d09e4d22a16c89450cb2b7f751ce", (
    "Prediction bytes differ; retain artifacts and diagnose runtime/numerics, never retune.",prediction_sha256)


Persisted pIC50: 1/6 fits
Persisted Caco-2: 2/6 fits
Persisted CYP3A4: 3/6 fits
Persisted hERG: 4/6 fits
Persisted HOB: 5/6 fits
Persisted MN: 6/6 fits


## 9. Evaluate the already-used holdout, without hiding denominators

Activity is evaluated on the original pIC50 scale: **RMSE** is primary; MAE and R² are secondary. For annotations, **balanced accuracy at 0.5** is primary; report both class recalls, AP for raw1 and raw0, ROC-AUC, MCC, Brier score, prevalence and all four confusion counts. AP0 uses `1 - p(raw1)`.

The small metric functions below are the same arithmetic definitions used in the audited evaluation. Tied score groups stay together for average precision and receive average ranks for AUC. Missing true classes produce `None` for undefined discrimination metrics rather than a misleading number. MCC is zero for constant calls only when both true classes exist. AP can equal 1 when only its queried class exists; that is not discrimination evidence. No confidence interval, significance claim or overall multi-endpoint score is added.


In [13]:
def activity_metrics(y,p):
    assert len(y)==len(p)>0 and all(math.isfinite(v) for v in y+p)
    n=len(y);mean=sum(y)/n;sst=sum((v-mean)**2 for v in y);sse=sum((a-b)**2 for a,b in zip(y,p))
    return dict(n=n,RMSE=math.sqrt(sse/n),MAE=sum(abs(a-b) for a,b in zip(y,p))/n,
        R2=1-sse/sst if n>=2 and sst>0 else None,
        undefined_reasons={} if n>=2 and sst>0 else {'R2':'fewer_than_two_rows_or_zero_target_variance'})

def ap(y,p):
    positives=sum(y)
    if positives==0:return None
    order=sorted(range(len(y)),key=lambda i:-p[i]);seen=tp=0;total=0.0;i=0
    while i<len(order):
        k=i+1
        while k<len(order) and p[order[k]]==p[order[i]]:k+=1
        added=sum(y[order[j]] for j in range(i,k));tp+=added;seen+=k-i
        total+=(added/positives)*(tp/seen);i=k
    return total

def auc(y,p):
    n1=sum(y);n0=len(y)-n1
    if not n0 or not n1:return None
    order=sorted(range(len(y)),key=lambda i:p[i]);rank_sum=0.0;i=0
    while i<len(order):
        k=i+1
        while k<len(order) and p[order[k]]==p[order[i]]:k+=1
        rank_sum+=sum(y[order[j]] for j in range(i,k))*((i+1+k)/2);i=k
    return (rank_sum-n1*(n1+1)/2)/(n0*n1)

def binary_metrics(y,p):
    assert len(y)==len(p)>0 and set(y)<={0,1} and all(math.isfinite(v) and 0<=v<=1 for v in p)
    pred=[int(v>=.5) for v in p];n=len(y);n1=sum(y);n0=n-n1
    tn=sum(a==0 and z==0 for a,z in zip(y,pred));fp=sum(a==0 and z==1 for a,z in zip(y,pred))
    fn=sum(a==1 and z==0 for a,z in zip(y,pred));tp=sum(a==1 and z==1 for a,z in zip(y,pred))
    r0=tn/n0 if n0 else None;r1=tp/n1 if n1 else None
    den=(tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)
    mcc=((tp*tn-fp*fn)/math.sqrt(den) if den else 0.0) if n0 and n1 else None
    undefined={}
    if not n0:undefined.update(recall_raw_0='true_raw0_absent',average_precision_raw_0='true_raw0_absent')
    if not n1:undefined.update(recall_raw_1='true_raw1_absent',average_precision_raw_1='true_raw1_absent')
    if not n0 or not n1:
        for k in ['balanced_accuracy','roc_auc_raw_1','mcc']:undefined[k]='both_true_classes_required'
    return dict(n=n,n_raw_0=n0,n_raw_1=n1,prevalence_raw_1=n1/n,
        balanced_accuracy=(r0+r1)/2 if n0 and n1 else None,
        average_precision_raw_1=ap(y,p),average_precision_raw_0=ap([1-v for v in y],[1-v for v in p]),
        roc_auc_raw_1=auc(y,p),mcc=mcc,brier_raw_1=sum((a-z)**2 for a,z in zip(y,p))/n,
        tn_raw_0=tn,fp_raw_0=fp,fn_raw_1=fn,tp_raw_1=tp,recall_raw_0=r0,recall_raw_1=r1,
        undefined_reasons=undefined,mcc_constant_prediction_convention='zero_if_both_true_classes_present')


In [14]:
# Historical, independently audited reference values; not notebook-generated outputs.
AUDITED_REFERENCE = {'activity_metrics': [{'model': 'fixed_RF',
                       'n': 395,
                       'RMSE': 0.7690156903852011,
                       'MAE': 0.5635407211191795,
                       'R2': 0.7004851373987786,
                       'undefined_reasons': {}},
                      {'model': 'development_mean',
                       'n': 395,
                       'RMSE': 1.4059721964800198,
                       'MAE': 1.168007912116104,
                       'R2': -0.0011552775700696127,
                       'undefined_reasons': {}}],
 'annotation_metrics': [{'endpoint': 'Caco-2',
                         'model': 'fixed_RF',
                         'n': 395,
                         'n_raw_0': 255,
                         'n_raw_1': 140,
                         'prevalence_raw_1': 0.35443037974683544,
                         'balanced_accuracy': 0.8838235294117647,
                         'average_precision_raw_1': 0.9267264899764825,
                         'average_precision_raw_0': 0.971909048221764,
                         'roc_auc_raw_1': 0.9528991596638655,
                         'mcc': 0.7676470588235295,
                         'brier_raw_1': 0.08181466164046829,
                         'tn_raw_0': 234,
                         'fp_raw_0': 21,
                         'fn_raw_1': 21,
                         'tp_raw_1': 119,
                         'recall_raw_0': 0.9176470588235294,
                         'recall_raw_1': 0.85,
                         'undefined_reasons': {},
                         'mcc_constant_prediction_convention': 'zero_if_both_true_classes_present'},
                        {'endpoint': 'Caco-2',
                         'model': 'development_prior',
                         'n': 395,
                         'n_raw_0': 255,
                         'n_raw_1': 140,
                         'prevalence_raw_1': 0.35443037974683544,
                         'balanced_accuracy': 0.5,
                         'average_precision_raw_1': 0.35443037974683544,
                         'average_precision_raw_0': 0.6455696202531646,
                         'roc_auc_raw_1': 0.5,
                         'mcc': 0.0,
                         'brier_raw_1': 0.23022248520720878,
                         'tn_raw_0': 255,
                         'fp_raw_0': 0,
                         'fn_raw_1': 140,
                         'tp_raw_1': 0,
                         'recall_raw_0': 1.0,
                         'recall_raw_1': 0.0,
                         'undefined_reasons': {},
                         'mcc_constant_prediction_convention': 'zero_if_both_true_classes_present'},
                        {'endpoint': 'CYP3A4',
                         'model': 'fixed_RF',
                         'n': 395,
                         'n_raw_0': 106,
                         'n_raw_1': 289,
                         'prevalence_raw_1': 0.7316455696202532,
                         'balanced_accuracy': 0.8990500750799766,
                         'average_precision_raw_1': 0.9917568776609766,
                         'average_precision_raw_0': 0.9221889384270795,
                         'roc_auc_raw_1': 0.9760396944571391,
                         'mcc': 0.8106341479387777,
                         'brier_raw_1': 0.05236560963222629,
                         'tn_raw_0': 89,
                         'fp_raw_0': 17,
                         'fn_raw_1': 12,
                         'tp_raw_1': 277,
                         'recall_raw_0': 0.839622641509434,
                         'recall_raw_1': 0.9584775086505191,
                         'undefined_reasons': {},
                         'mcc_constant_prediction_convention': 'zero_if_both_true_classes_present'},
                        {'endpoint': 'CYP3A4',
                         'model': 'development_prior',
                         'n': 395,
                         'n_raw_0': 106,
                         'n_raw_1': 289,
                         'prevalence_raw_1': 0.7316455696202532,
                         'balanced_accuracy': 0.5,
                         'average_precision_raw_1': 0.7316455696202532,
                         'average_precision_raw_0': 0.2683544303797468,
                         'roc_auc_raw_1': 0.5,
                         'mcc': 0.0,
                         'brier_raw_1': 0.19645261282834653,
                         'tn_raw_0': 0,
                         'fp_raw_0': 106,
                         'fn_raw_1': 0,
                         'tp_raw_1': 289,
                         'recall_raw_0': 0.0,
                         'recall_raw_1': 1.0,
                         'undefined_reasons': {},
                         'mcc_constant_prediction_convention': 'zero_if_both_true_classes_present'},
                        {'endpoint': 'hERG',
                         'model': 'fixed_RF',
                         'n': 395,
                         'n_raw_0': 161,
                         'n_raw_1': 234,
                         'prevalence_raw_1': 0.5924050632911393,
                         'balanced_accuracy': 0.8949540797366884,
                         'average_precision_raw_1': 0.9749901021587652,
                         'average_precision_raw_0': 0.9532802980369841,
                         'roc_auc_raw_1': 0.9642458990285078,
                         'mcc': 0.786367727006485,
                         'brier_raw_1': 0.0745494530235255,
                         'tn_raw_0': 143,
                         'fp_raw_0': 18,
                         'fn_raw_1': 23,
                         'tp_raw_1': 211,
                         'recall_raw_0': 0.8881987577639752,
                         'recall_raw_1': 0.9017094017094017,
                         'undefined_reasons': {},
                         'mcc_constant_prediction_convention': 'zero_if_both_true_classes_present'},
                        {'endpoint': 'hERG',
                         'model': 'development_prior',
                         'n': 395,
                         'n_raw_0': 161,
                         'n_raw_1': 234,
                         'prevalence_raw_1': 0.5924050632911393,
                         'balanced_accuracy': 0.5,
                         'average_precision_raw_1': 0.5924050632911393,
                         'average_precision_raw_0': 0.40759493670886077,
                         'roc_auc_raw_1': 0.5,
                         'mcc': 0.0,
                         'brier_raw_1': 0.2434495715274043,
                         'tn_raw_0': 0,
                         'fp_raw_0': 161,
                         'fn_raw_1': 0,
                         'tp_raw_1': 234,
                         'recall_raw_0': 0.0,
                         'recall_raw_1': 1.0,
                         'undefined_reasons': {},
                         'mcc_constant_prediction_convention': 'zero_if_both_true_classes_present'},
                        {'endpoint': 'HOB',
                         'model': 'fixed_RF',
                         'n': 395,
                         'n_raw_0': 313,
                         'n_raw_1': 82,
                         'prevalence_raw_1': 0.20759493670886076,
                         'balanced_accuracy': 0.8330281306007948,
                         'average_precision_raw_1': 0.7402716973037081,
                         'average_precision_raw_0': 0.9790847481073433,
                         'roc_auc_raw_1': 0.9257967739421803,
                         'mcc': 0.632073496316136,
                         'brier_raw_1': 0.09019713791668874,
                         'tn_raw_0': 281,
                         'fp_raw_0': 32,
                         'fn_raw_1': 19,
                         'tp_raw_1': 63,
                         'recall_raw_0': 0.8977635782747604,
                         'recall_raw_1': 0.7682926829268293,
                         'undefined_reasons': {},
                         'mcc_constant_prediction_convention': 'zero_if_both_true_classes_present'},
                        {'endpoint': 'HOB',
                         'model': 'development_prior',
                         'n': 395,
                         'n_raw_0': 313,
                         'n_raw_1': 82,
                         'prevalence_raw_1': 0.20759493670886076,
                         'balanced_accuracy': 0.5,
                         'average_precision_raw_1': 0.20759493670886076,
                         'average_precision_raw_0': 0.7924050632911392,
                         'roc_auc_raw_1': 0.5,
                         'mcc': 0.0,
                         'brier_raw_1': 0.16844681026458577,
                         'tn_raw_0': 313,
                         'fp_raw_0': 0,
                         'fn_raw_1': 82,
                         'tp_raw_1': 0,
                         'recall_raw_0': 1.0,
                         'recall_raw_1': 0.0,
                         'undefined_reasons': {},
                         'mcc_constant_prediction_convention': 'zero_if_both_true_classes_present'},
                        {'endpoint': 'MN',
                         'model': 'fixed_RF',
                         'n': 395,
                         'n_raw_0': 95,
                         'n_raw_1': 300,
                         'prevalence_raw_1': 0.759493670886076,
                         'balanced_accuracy': 0.872280701754386,
                         'average_precision_raw_1': 0.9958353918056954,
                         'average_precision_raw_0': 0.9608407382752169,
                         'roc_auc_raw_1': 0.9865964912280701,
                         'mcc': 0.807273714808692,
                         'brier_raw_1': 0.04731578183070387,
                         'tn_raw_0': 72,
                         'fp_raw_0': 23,
                         'fn_raw_1': 4,
                         'tp_raw_1': 296,
                         'recall_raw_0': 0.7578947368421053,
                         'recall_raw_1': 0.9866666666666667,
                         'undefined_reasons': {},
                         'mcc_constant_prediction_convention': 'zero_if_both_true_classes_present'},
                        {'endpoint': 'MN',
                         'model': 'development_prior',
                         'n': 395,
                         'n_raw_0': 95,
                         'n_raw_1': 300,
                         'prevalence_raw_1': 0.759493670886076,
                         'balanced_accuracy': 0.5,
                         'average_precision_raw_1': 0.759493670886076,
                         'average_precision_raw_0': 0.24050632911392406,
                         'roc_auc_raw_1': 0.5,
                         'mcc': 0.0,
                         'brier_raw_1': 0.18275040805384732,
                         'tn_raw_0': 0,
                         'fp_raw_0': 95,
                         'fn_raw_1': 0,
                         'tp_raw_1': 300,
                         'recall_raw_0': 0.0,
                         'recall_raw_1': 1.0,
                         'undefined_reasons': {},
                         'mcc_constant_prediction_convention': 'zero_if_both_true_classes_present'}]}
INDEPENDENT_FINAL_REVIEW_SHA256 = '10866bc7fcdd9162705a1907dfe6ceb6a62db7544dac1e17dec3778d8e0ad8ce'

def compare_reference(actual, expected, trail="root"):
    if isinstance(expected,dict):
        assert set(actual)==set(expected), trail
        for key in expected: compare_reference(actual[key],expected[key],trail+"."+key)
    elif isinstance(expected,list):
        assert len(actual)==len(expected),trail
        for i,(a,e) in enumerate(zip(actual,expected)): compare_reference(a,e,f"{trail}[{i}]")
    elif isinstance(expected,float):
        assert math.isclose(actual,expected,rel_tol=0,abs_tol=1e-12),(trail,actual,expected)
    else:
        assert actual==expected,(trail,actual,expected)


In [15]:
require_replay()
assert completed_fits==6 and prediction_sha256=="baef49de75f18b21f6320900218481beb3e0d09e4d22a16c89450cb2b7f751ce"
write_json(RUN_DIR / "evaluation_started.json",{"holdout_status":"already_used_replay",
    "prediction_sha256":prediction_sha256,"rows":395})
y_holdout = [float(r["pIC50"]) for r in holdout]
activity_results = [
    {"model":"fixed_RF",**activity_metrics(y_holdout,[r["predicted_source_pIC50"] for r in predictions])},
    {"model":"development_mean",**activity_metrics(y_holdout,[activity_mean]*395)}]
annotation_results = []
for endpoint, output in zip(ENDPOINTS,OUTPUTS):
    y = [r[endpoint] for r in holdout]
    annotation_results.extend([
        {"endpoint":endpoint,"model":"fixed_RF",**binary_metrics(y,[r[output] for r in predictions])},
        {"endpoint":endpoint,"model":"development_prior",**binary_metrics(y,[priors[endpoint]]*395)}])
results = {"activity_metrics":activity_results,"annotation_metrics":annotation_results}
write_json(RUN_DIR / "replayed_metrics.json",results)
compare_reference(results,AUDITED_REFERENCE)

# Keep observed/source outcomes separate from calculated predictions, joined by stable ID.
comparison = []
for source,prediction in zip(holdout,predictions):
    assert source["source_record_id"]==prediction["source_record_id"]
    comparison.append({**prediction,"source_reported_IC50_nM":source["IC50_nM"],
        "source_supplied_pIC50":source["pIC50"],
        **{"source_annotation_"+e:source[e] for e in ENDPOINTS}})
write_csv(RUN_DIR / "predictions_and_source_outcomes.csv",comparison)
print(json.dumps(results,indent=2))
print("Exact prediction bytes and all reference metrics reproduced. This was not a fresh validation.")


{
  "activity_metrics": [
    {
      "model": "fixed_RF",
      "n": 395,
      "RMSE": 0.7690156903852011,
      "MAE": 0.5635407211191795,
      "R2": 0.7004851373987786,
      "undefined_reasons": {}
    },
    {
      "model": "development_mean",
      "n": 395,
      "RMSE": 1.4059721964800198,
      "MAE": 1.168007912116104,
      "R2": -0.0011552775700696127,
      "undefined_reasons": {}
    }
  ],
  "annotation_metrics": [
    {
      "endpoint": "Caco-2",
      "model": "fixed_RF",
      "n": 395,
      "n_raw_0": 255,
      "n_raw_1": 140,
      "prevalence_raw_1": 0.35443037974683544,
      "balanced_accuracy": 0.8838235294117647,
      "average_precision_raw_1": 0.9267264899764825,
      "average_precision_raw_0": 0.971909048221764,
      "roc_auc_raw_1": 0.9528991596638655,
      "mcc": 0.7676470588235295,
      "brier_raw_1": 0.08181466164046829,
      "tn_raw_0": 234,
      "fp_raw_0": 21,
      "fn_raw_1": 21,
      "tp_raw_1": 119,
      "recall_raw_0": 0.91764705882

## 10. Inspect a prediction, then keep the limitations with the result

The next cell displays the **first five holdout rows in frozen order**, not hand-picked successes. Source-reported IC50 and source-supplied pIC50 are outcomes. Every `p_raw1_...` field and `predicted_source_pIC50` is a model prediction. All 395 comparisons are saved, including errors.

The fitted six-model candidate is limited to prediction of the supplied outcomes. Unknown annotation provenance cannot be repaired by high classification scores. Descriptors and RF scores do not establish mechanism, dose-response toxicity, a therapeutic index, calibrated uncertainty or clinical safety.

No model is fitted to all 1,974 labeled rows afterward. The 50 unlabeled source records and any separate registry are not predicted. There is no arbitrary-molecule prediction interface or explorer integration in this artifact. Any later model/feature/threshold redesign must treat this holdout as used evidence and obtain genuinely new evaluation data for a fresh accuracy claim.


In [16]:
require_replay()
from html import escape
try:
    from IPython.display import HTML, display
except ImportError:
    HTML = display = None  # Bare Python replay needs no notebook-interface installation.

def show_table(rows, columns):
    if display is None:
        text = io.StringIO()
        writer = csv.DictWriter(text,fieldnames=columns,extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)
        print(text.getvalue())
        return
    header = "".join("<th>"+escape(c)+"</th>" for c in columns)
    body = "".join("<tr>"+"".join("<td>"+escape(str(r[c]))+"</td>" for c in columns)+"</tr>" for r in rows)
    display(HTML("<div style='overflow-x:auto'><table><thead><tr>"+header+"</tr></thead><tbody>"+body+"</tbody></table></div>"))

show_table(comparison[:5],["source_record_id","source_reported_IC50_nM","source_supplied_pIC50","predicted_source_pIC50"])
for endpoint,output in zip(ENDPOINTS,OUTPUTS):
    show_table(comparison[:5],["source_record_id","source_annotation_"+endpoint,output,"raw_call_"+endpoint])
write_json(RUN_DIR / "reproduction_complete.json",{"status":"FIXED_USED_HOLDOUT_REPRODUCED",
    "completed_fits":6,"rows":395,"prediction_sha256":prediction_sha256,
    "metric_absolute_tolerance":1e-12,"new_independent_validation":False,
    "protocol_sha256":FROZEN_PROTOCOL_SHA256,"reference_audit_sha256":INDEPENDENT_FINAL_REVIEW_SHA256})
manifest = {p.name:digest_bytes(p.read_bytes()) for p in sorted(RUN_DIR.iterdir()) if p.is_file()}
write_json(RUN_DIR / "artifact_manifest.json",manifest)


source_record_id,source_reported_IC50_nM,source_supplied_pIC50,predicted_source_pIC50
figshare30459479v2:training:row34,3.6,8.44369749923271,8.531595575443962
figshare30459479v2:training:row40,94.0,7.0268721464003,7.715352410990974
figshare30459479v2:training:row48,4.0,8.39794000867204,8.08206404344161
figshare30459479v2:training:row49,5.0,8.30102999566398,8.057345695482116
figshare30459479v2:training:row50,4.0,8.39794000867204,8.302860380232092

source_record_id,source_annotation_Caco-2,p_raw1_Caco2,raw_call_Caco-2
figshare30459479v2:training:row34,0,0.011944444444444443,0
figshare30459479v2:training:row40,0,0.013888888888888886,0
figshare30459479v2:training:row48,0,0.005277777777777778,0
figshare30459479v2:training:row49,0,0.005277777777777778,0
figshare30459479v2:training:row50,0,0.009722222222222222,0

source_record_id,source_annotation_CYP3A4,p_raw1_CYP3A4_metabolized,raw_call_CYP3A4
figshare30459479v2:training:row34,1,0.9694444444444443,1
figshare30459479v2:training:row40,1,0.995

### Audit anchors and what is still untested

The accepted protocol digest is `cb26d3242d9c2dcd7b7d87f054e9285edad64934a7a974f07804baaf55800fe7`; terminal independent audit digest is `10866bc7fcdd9162705a1907dfe6ceb6a62db7544dac1e17dec3778d8e0ad8ce`. These identify the original local records; they are not public download links and are not needed as input files. Reference results in this notebook were copied from that audit chain. The underlying official source files are retrieved by fixed public IDs and verified before use.

The source notebook was first independently reviewed without execution, then replayed once under a separate exact-replay authorization. The terminal independent audit accepted the actual cell outputs, source/feature/split checks, six fixed fits, exact original prediction bytes and reproduced metrics. This publication copy changes only historical packaging-state prose and metadata; all executed code, outputs and counts are preserved. The execution engine was sequential bare CPython 3.13.16, not a native Jupyter/Colab front-end test. Audit and source hashes are recorded in the notebook metadata. This remains reproduction of already-used holdout evidence, not fresh validation.
